# Data Source 01 · 금융·투자

**목적:** 수집할 시계열의 단위·주기·결측·발표시점을 점검한다.

| 출처 | 접근 | 검증 상태 |
|---|---|---|
| FRED `fredgraph.csv` (예: `DGS10` 10년 국채금리) | 무키 | B (엔드포인트·헤더 수업 전 확인) |
| Yahoo Finance (`yfinance`) | 무키(비공식) | B |
| 한국은행 ECOS | 무료키 | C |
| Ken French Data Library | 수동 | C |

자세한 목록: `resources/Data_Source_Catalog.md`. **API 키는 파일에 쓰지 말고 `getpass`/환경변수로만 입력**한다.
온라인 연결이 실패하면 합성자료로 대체하며 `DATA_KIND`로 구분한다. 합성자료의 결과는 실제 현상의 증거가 아니다.

In [ ]:
import io, json, urllib.request, numpy as np, pandas as pd, matplotlib.pyplot as plt

def fetch_bytes(url, timeout=30):
    """무키 공개 URL 요청. 실패 시 예외 -> 호출부에서 합성 대체."""
    req = urllib.request.Request(url, headers={"User-Agent": "ts-lab-course/1.0 (education)"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.read()

def synthetic_series(n, freq, seed, trend=0.05, season=12, noise=1.0, start="2015-01-01"):
    """NOT real data: 방법 시연용 합성 시계열."""
    rng = np.random.default_rng(seed)
    t = np.arange(n)
    y = 100 + trend * t + 3 * np.sin(2 * np.pi * t / season) + rng.normal(0, noise, n)
    return pd.Series(y, index=pd.date_range(start, periods=n, freq=freq), name="value")

def quality_report(s):
    s = s.sort_index()
    print("관측 수:", len(s), "| 기간:", s.index.min(), "~", s.index.max())
    print("결측:", int(s.isna().sum()), "| 중복 시점:", int(s.index.duplicated().sum()))
    f = pd.infer_freq(s.index) if len(s) > 2 else None
    print("추정 주기:", f)

In [ ]:
def parse_fred_csv(raw):
    """FRED fredgraph.csv 파싱. 날짜 열 이름이 DATE 또는 observation_date일 수 있고, 결측은 '.' 로 표시될 수 있다."""
    df = pd.read_csv(io.BytesIO(raw) if isinstance(raw, (bytes, bytearray)) else io.StringIO(raw))
    date_col = next(c for c in df.columns if c.lower() in ("date", "observation_date"))
    val_col = [c for c in df.columns if c != date_col][0]
    s = pd.Series(pd.to_numeric(df[val_col], errors="coerce").values,
                  index=pd.to_datetime(df[date_col]), name=val_col)
    return s

In [ ]:
SERIES_ID = "DGS10"   # FRED 시리즈 ID. 실제 존재·단위는 FRED 페이지에서 확인할 것.
try:
    raw = fetch_bytes(f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={SERIES_ID}")
    y = parse_fred_csv(raw)
    DATA_KIND = "REAL: FRED " + SERIES_ID
except Exception as e:
    print("온라인 수집 실패 -> 합성 대체:", type(e).__name__, e)
    y = synthetic_series(1500, "B", 2026011, trend=0.002, season=250, noise=0.15, start="2018-01-01")
    DATA_KIND = "SYNTHETIC (NOT real data)"
print("DATA_KIND:", DATA_KIND)
quality_report(y)
y.plot(title=DATA_KIND); plt.show()

## 점검 체크리스트
- 단위(%, 지수, 수준)와 **주기**(영업일/월)가 연구질문과 맞는가
- 결측(휴장일)·대체 규칙을 사전에 정의했는가
- 수정주가/발표시점(빈티지): 사후 개정 값을 쓰면 look-ahead 위험
- 라이선스·재배포 조건: 원자료는 저장소에 올리지 않는다

## 추가 A. 미국 거시 3종 (FRED: FEDFUNDS, UNRATE, CPIAUCSL)
- 원 출처: 학생 예제 03번(`pandas_datareader`). 여기서는 **무키 `fredgraph.csv`** 로 읽어 라이브러리 의존을 줄였다(실행 확인 필요).
- 물가상승률은 **월별로 정렬한 뒤** `pct_change(12)`로 계산한다(원 예제는 정렬 전에 계산).
- CPIAUCSL은 사후 개정되는 빈티지 자료이므로 예측 평가에서는 look-ahead 위험을 명시한다.

In [ ]:
def build_macro_panel(series_dict):
    """series_dict: {이름: pd.Series}. 월 시작(MS) 기준으로 맞추고 CPI에서 전년동월 대비 상승률을 만든다."""
    df = pd.concat(series_dict, axis=1).sort_index()
    df = df.resample("MS").mean()
    if "CPI" in df:
        df["INFL_YOY"] = df["CPI"].pct_change(12) * 100
    return df

def check_macro(df):
    assert df["FEDFUNDS"].dropna().between(0, 25).all(), "금리 범위 이상"
    assert df["UNRATE"].dropna().between(1, 30).all(), "실업률 범위 이상"

In [ ]:
MACRO_IDS = {"FEDFUNDS": "FEDFUNDS", "UNRATE": "UNRATE", "CPI": "CPIAUCSL"}
try:
    ser = {k: parse_fred_csv(fetch_bytes(f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={v}")) for k, v in MACRO_IDS.items()}
    macro = build_macro_panel(ser).loc["2000-01-01":"2024-12-31"]
    check_macro(macro)
    KIND_MACRO = "REAL: FRED " + ", ".join(MACRO_IDS.values())
except Exception as e:
    print("FRED 거시 수집 실패 -> 합성 대체:", type(e).__name__, str(e)[:100])
    n = 300; rng = np.random.default_rng(2026071); idx = pd.date_range("2000-01-01", periods=n, freq="MS")
    ff = np.clip(2 + np.cumsum(rng.normal(0, 0.15, n)), 0.05, 8)
    un = np.clip(5.5 + np.cumsum(rng.normal(0, 0.12, n)), 3, 12)
    cpi = 170 * np.cumprod(1 + rng.normal(0.0018, 0.002, n))
    macro = build_macro_panel({"FEDFUNDS": pd.Series(ff, idx), "UNRATE": pd.Series(un, idx), "CPI": pd.Series(cpi, idx)})
    KIND_MACRO = "SYNTHETIC (NOT real data)"
print("KIND_MACRO:", KIND_MACRO); print(macro.describe().round(2).T[["count", "mean", "min", "max"]])
macro[["FEDFUNDS", "UNRATE", "INFL_YOY"]].plot(subplots=True, figsize=(10, 6), title=KIND_MACRO); plt.show()

## 추가 B. 금 가격 (저장소 포함 CSV `data/finance/gold_historical_data.csv`)
- 학생 예제 06번이 쓰는 파일을 저장소에 포함했다(2016-02-05 ~ 2026-01-30, 2,510행, 열: Date, Adj Close, Close, High, Low, Open, Volume).
- **출처·라이선스 표기가 없는 파일이다.** Adj Close = Close, 거래량 0 행 36개, 가격대로 보아 Yahoo Finance 금 선물(`GC=F`) 계열로 **추정**되나 확인 필요하며, 재배포 가능 여부도 확인 필요하다.
- 읽는 순서: ① 저장소 파일(로컬 경로 또는 GitHub raw) → ② `yfinance` `GC=F` → ③ 합성. `KIND_GOLD`로 구분한다.
- 금은 금융 시계열이다(비주식이지만 비금융이 아니다). 연속월물 롤오버 단절 가능성과 거래량 0 행은 분석 전에 처리 규칙을 정한다.

In [ ]:
def load_gold_csv(raw):
    df = pd.read_csv(io.BytesIO(raw) if isinstance(raw, (bytes, bytearray)) else io.StringIO(raw), parse_dates=["Date"])
    s = df.set_index("Date")["Close"].astype(float).sort_index().rename("gold")
    assert len(s) > 1500 and s.min() > 100 and not s.index.duplicated().any(), "금 CSV 길이/범위/중복 이상"
    return s

def to_log_return(price):
    return np.log(price).diff().dropna().rename("ret")

import os
GOLD_LOCAL = [p for p in ("data/finance/gold_historical_data.csv", "../data/finance/gold_historical_data.csv",
                          "gold_historical_data.csv") if os.path.exists(p)]
GOLD_RAW = "https://raw.githubusercontent.com/nhjung-phd/TimeSeriesResearchMethods/main/data/finance/gold_historical_data.csv"  # push 후 존재, 확인 필요
gold, KIND_GOLD = None, None
try:
    if GOLD_LOCAL:
        gold = load_gold_csv(open(GOLD_LOCAL[0], "rb").read()); KIND_GOLD = "REAL(file): " + GOLD_LOCAL[0] + " (출처 추정 GC=F)"
    else:
        gold = load_gold_csv(fetch_bytes(GOLD_RAW)); KIND_GOLD = "REAL(file): GitHub raw (출처 추정 GC=F)"
except Exception as e:
    print("금 CSV 읽기 실패 -> yfinance 시도:", type(e).__name__, str(e)[:80])
if gold is None:
    try:
        import subprocess, sys, importlib.util
        if importlib.util.find_spec("yfinance") is None:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "yfinance"])
        import yfinance as yf
        g = yf.download("GC=F", start="2016-02-01", end="2026-02-01", progress=False, auto_adjust=False)
        gold = g["Close"].squeeze().dropna().rename("gold")
        assert len(gold) > 1500 and gold.min() > 100
        KIND_GOLD = "REAL: Yahoo Finance GC=F (연속월물, 비공식 API)"
    except Exception as e:
        print("금 yfinance 실패 -> 합성 대체:", type(e).__name__, str(e)[:80])
        rng = np.random.default_rng(2026072)
        gold = pd.Series(1200 * np.exp(np.cumsum(rng.normal(0.0003, 0.01, 2500))),
                         index=pd.bdate_range("2016-02-05", periods=2500), name="gold")
        KIND_GOLD = "SYNTHETIC (NOT real data)"
print("KIND_GOLD:", KIND_GOLD); quality_report(gold)
r = to_log_return(gold); print("연율 변동성:", round(float(r.std() * np.sqrt(252)), 3))